# Neuro-Symbolic Concept Learning

## What This Is
NSCL (Mao et al., 2019) decomposes visual reasoning into two stages:
1. **Neural perception module**: extract object features from images, segment objects
2. **Symbolic reasoning module**: execute a program over the extracted objects to answer questions

This enables compositional generalization — the system can answer new types of questions that recombine known concepts.

## Example: CLEVR-style visual QA
- Input: image of colored shapes + question "What is the color of the large cube?"
- Neural module: detect objects, extract attributes (shape, color, size, material)
- Program: filter(shape=cube) → filter(size=large) → query(color)
- Symbolic execution: run the program on the object set

## Why It Works
Neural module handles perception uncertainty; symbolic module handles compositional reasoning. The split enables:
- Systematic generalization to new question types
- Interpretable reasoning traces
- Data-efficient learning (symbolic module generalizes from few examples)

In [ ]:
import numpy as np
from typing import List, Dict, Optional, Tuple

np.random.seed(42)

# NSCL-style pipeline: perception + symbolic reasoning

# Object representation
COLORS = ['red', 'blue', 'green', 'yellow', 'purple']
SHAPES = ['cube', 'sphere', 'cylinder']
SIZES = ['small', 'large']
MATERIALS = ['metal', 'rubber']

class SceneObject:
    def __init__(self, color, shape, size, material, x, y):
        self.color = color
        self.shape = shape
        self.size = size
        self.material = material
        self.pos = (x, y)
    
    def __repr__(self):
        return f'{self.size} {self.color} {self.material} {self.shape}'

def generate_scene(n_objects=5) -> List[SceneObject]:
    objects = []
    for _ in range(n_objects):
        objects.append(SceneObject(
            color=np.random.choice(COLORS),
            shape=np.random.choice(SHAPES),
            size=np.random.choice(SIZES),
            material=np.random.choice(MATERIALS),
            x=np.random.uniform(0, 10),
            y=np.random.uniform(0, 10),
        ))
    return objects

# Symbolic execution engine
class SymbolicReasoner:
    def execute(self, program: List[Dict], objects: List[SceneObject]):
        current = list(objects)  # start with all objects
        
        for step in program:
            op = step['op']
            if op == 'filter_color':
                current = [o for o in current if o.color == step['value']]
            elif op == 'filter_shape':
                current = [o for o in current if o.shape == step['value']]
            elif op == 'filter_size':
                current = [o for o in current if o.size == step['value']]
            elif op == 'filter_material':
                current = [o for o in current if o.material == step['value']]
            elif op == 'query_color':
                return [o.color for o in current]
            elif op == 'query_shape':
                return [o.shape for o in current]
            elif op == 'query_size':
                return [o.size for o in current]
            elif op == 'count':
                return len(current)
            elif op == 'exist':
                return len(current) > 0
            elif op == 'unique':
                if len(current) == 1: return current[0]
                return None
        return current

# Perception module: classify object attributes from noisy features
class PerceptionModule:
    def perceive(self, obj: SceneObject, noise: float = 0.1) -> Dict:
        """Simulate noisy perception of object attributes."""
        # In reality: CNN feature extractor
        perceived_color = obj.color if np.random.random() > noise else np.random.choice(COLORS)
        perceived_shape = obj.shape if np.random.random() > noise else np.random.choice(SHAPES)
        perceived_size = obj.size if np.random.random() > noise*0.5 else np.random.choice(SIZES)
        perceived_material = obj.material if np.random.random() > noise*2 else np.random.choice(MATERIALS)
        return {'color': perceived_color, 'shape': perceived_shape,
                'size': perceived_size, 'material': perceived_material}

# Test the pipeline
scene = generate_scene(5)
perception = PerceptionModule()
reasoner = SymbolicReasoner()

print('Generated scene:')
for i, obj in enumerate(scene):
    print(f'  Object {i}: {obj}')


In [ ]:
# Question answering via program execution

test_programs = [
    {
        'question': 'How many cubes are in the scene?',
        'program': [{'op': 'filter_shape', 'value': 'cube'}, {'op': 'count'}]
    },
    {
        'question': 'What color is the large sphere?',
        'program': [{'op': 'filter_shape', 'value': 'sphere'},
                    {'op': 'filter_size', 'value': 'large'},
                    {'op': 'query_color'}]
    },
    {
        'question': 'Is there a red metal cube?',
        'program': [{'op': 'filter_color', 'value': 'red'},
                    {'op': 'filter_material', 'value': 'metal'},
                    {'op': 'filter_shape', 'value': 'cube'},
                    {'op': 'exist'}]
    },
]

print('Visual QA Results:')
for qa in test_programs:
    result = reasoner.execute(qa['program'], scene)
    print(f'  Q: {qa["question"]}')
    print(f'  A: {result}')
    print()

# Demonstrate perception → reasoning pipeline
print('Perception + Reasoning Pipeline (with noise):')
n_correct = 0
n_total = 100

for _ in range(n_total):
    test_scene = generate_scene(4)
    # Ground truth
    true_result = reasoner.execute([{'op': 'filter_shape', 'value': 'cube'}, {'op': 'count'}], test_scene)
    
    # Perceived scene (with noise)
    perceived_scene = []
    for obj in test_scene:
        p = perception.perceive(obj, noise=0.15)
        perceived_scene.append(SceneObject(p['color'], p['shape'], p['size'], p['material'], 0, 0))
    
    pred_result = reasoner.execute([{'op': 'filter_shape', 'value': 'cube'}, {'op': 'count'}], perceived_scene)
    if pred_result == true_result:
        n_correct += 1

print(f'  Count accuracy under 15% perception noise: {n_correct/n_total:.3f}')
print('  Key insight: symbolic execution is brittle to perception errors — use soft predicates in NTP')
